# HolidayCheck Reviews – Debug & Test Notebook

Step-by-step testing for `src/sites/holidaycheck_reviews.py`.

Covers:
1. Web scraping – fetching reviews from HolidayCheck hotel pages (Playwright)
2. Section extraction – full review text from all topics (Allgemein, Zimmer, Service, etc.)
3. Ollama classification – topic & sentiment detection
4. Manual review input – adding reviews that the scraper can't fetch

**Requirements:** Ollama running locally with `qwen2.5:7b`, Playwright installed.
No API key needed – HolidayCheck reviews are scraped from the public website.

In [1]:
from pathlib import Path
import sys, os, json

ROOT = Path.cwd()
if not (ROOT / 'src').exists() and (ROOT.parent / 'src').exists():
    ROOT = ROOT.parent

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print('Project root:', ROOT)

Project root: /home/laurabquintas/projects/reputation-analyzer


In [2]:
from src.sites import holidaycheck_reviews as hcr
import requests  # used for Ollama API calls

hotel_url = hcr.HC_URLS.get(hcr.ANANEA_HOTEL)
reviews_url = hcr._hotel_url_to_reviews_url(hotel_url)
print(f'Hotel: {hcr.ANANEA_HOTEL}')
print(f'Hotel URL: {hotel_url}')
print(f'Reviews URL: {reviews_url}')

Hotel: Ananea Castelo Suites Hotel
Hotel URL: https://www.holidaycheck.de/hi/ananea-castelo-suites-algarve/069563af-47db-44a3-bdb1-3441ae3a2ac4
Reviews URL: https://www.holidaycheck.de/hr/bewertungen-ananea-castelo-suites-algarve/069563af-47db-44a3-bdb1-3441ae3a2ac4


## 1. Web Scraping – Single Detail Page (Playwright)

Fetch the listing page sorted by newest, pick one review detail link,
and extract the full review text including all topic sections.

In [3]:
# Fetch just 1 page of reviews (auto-detects Jupyter and uses async Playwright)
single_page_reviews = hcr.hc_get_reviews(hotel_url, max_pages=1, min_delay=1.0, max_delay=2.0, sort_newest=True)
print(f"Fetched {len(single_page_reviews)} reviews from page 1 (sorted by newest)\n")

# Show the first review with full section detail
if single_page_reviews:
    review = single_page_reviews[0]
    rating = hcr._normalize_rating(review.get('rating')) or '?'
    print(f"Title: {review.get('title', '(none)')}")
    print(f"Rating: {rating}/6")
    print(f"Date: {review.get('travel_date', '?')}")
    print(f"Author: {review.get('author_name', '?')}")
    print(f"Country: {review.get('country', '(unknown)')}")
    print(f"Trip type: {review.get('trip_type', '(unknown)')}")
    print(f"Text length: {len(review.get('text', ''))} chars")

    # Show which sections were extracted
    text = review.get('text', '')
    sections = [line.split(']')[0].strip('[') for line in text.split('\n') if line.startswith('[')]
    if sections:
        print(f"Sections: {sections}")

    print(f"\nFull text:\n{text}")

Fetched 10 reviews from page 1 (sorted by newest)

Title: Top-Hotel
Rating: 5.0/6
Date: 2026-03-19
Author: Daniel
Country: Unknown
Trip type: Unknown
Text length: 176 chars
Sections: ['Zimmer', 'Service', 'Restaurant & Bars']

Full text:
[Zimmer] ZimmergrößeGutSchlafqualitätSehr gutSauberkeitSehr gut

[Service] Bester ServiceBar, Disco & RestaurantGästebetreuung

[Restaurant & Bars] EssensauswahlGutGeschmackGut


## 2. Web Scraping – All Pages with Pagination (Sorted by Newest)

Uses `hc_get_reviews()` with Playwright to fetch all pages sorted by newest.
Each review is fetched from its detail page to get the full text across all
topic sections (Allgemein, Zimmer, Service, Lage, etc.).

In [4]:
# Fetch reviews across multiple pages (sorted by newest, using Playwright)
all_reviews = hcr.hc_get_reviews(hotel_url, max_pages=5, min_delay=1.0, max_delay=2.0, sort_newest=True)
print(f'Total unique reviews fetched: {len(all_reviews)}')
print()
for r in all_reviews:
    rating = hcr._normalize_rating(r.get('rating')) or '?'
    country = r.get('country', '')
    trip = r.get('trip_type', '')
    meta = f" | {country}" if country else ""
    meta += f" | {trip}" if trip else ""
    sections = [line.split(']')[0].strip('[') for line in r.get('text', '').split('\n') if line.startswith('[')]
    section_str = f" [{', '.join(sections)}]" if sections else ""
    print(f"  [{r.get('id', 'N/A')[:16]}] {rating}/6 – {r.get('title', '(no title)')[:50]} ({r.get('travel_date', '')}){meta}{section_str}")

Total unique reviews fetched: 20

  [cc31415556e56b32] 5.0/6 – Top-Hotel (2026-03-19) | Unknown | Unknown [Zimmer, Service, Restaurant & Bars]
  [5c353f2cb8bc3b57] 6.0/6 – Modernes neues Hotel mit traumhaftem Rooftop-Pool (2026-02-23) | Unknown | Unknown
  [eeab75500d11bb6d] 6.0/6 – insgesamt empfehlenswert (2025-11-23) | Unknown | Unknown
  [2d39c4bae1831f24] 6.0/6 – Neues stilvolles Hotel (2025-11-12) | Unknown | Unknown [Zimmer, Service, Lage & Umgebung]
  [e62885c6c5699296] 6.0/6 – Hotel kann man weiterempfehlen (2025-11-03) | Unknown | Unknown [Zimmer, Service, Lage & Umgebung, Restaurant & Bars]
  [6beb650dbd0723de] 6.0/6 – Erholsame Urlaubstage in einem neuen schönen Hotel (2025-10-30) | Unknown | Unknown [Zimmer, Service, Lage & Umgebung, Restaurant & Bars]
  [57455503d9dd158d] 4.0/6 – Neues Hotel mit Potential nach Oben (2025-10-30) | Unknown | Unknown
  [6aca7fceb54d45b9] 6.0/6 – Weil einfach alles gepasst hat (2025-10-17) | Unknown | Unknown [Zimmer, Service, Lage & Umgebung

In [5]:
# Inspect raw scraped data for first review (now includes section labels)
if all_reviews:
    r = all_reviews[0]
    print(json.dumps(r, indent=2, ensure_ascii=False))
    print()

    # Show which sections were extracted
    sections = [line.split(']')[0].strip('[') for line in r.get('text', '').split('\n') if line.startswith('[')]
    if sections:
        print(f"Sections extracted: {sections}")
    else:
        print("No section labels found (review may only have Allgemein)")

{
  "id": "cc31415556e56b32",
  "rating": 8.3,
  "title": "Top-Hotel",
  "text": "[Zimmer] ZimmergrößeGutSchlafqualitätSehr gutSauberkeitSehr gut\n\n[Service] Bester ServiceBar, Disco & RestaurantGästebetreuung\n\n[Restaurant & Bars] EssensauswahlGutGeschmackGut",
  "travel_date": "2026-03-19",
  "author_name": "Daniel",
  "country": "Unknown",
  "trip_type": "Unknown"
}

Sections extracted: ['Zimmer', 'Service', 'Restaurant & Bars']


## 3. Ollama Classification – Test

In [6]:
# Check Ollama availability
from src.classification import classify_holidaycheck_review, is_ollama_available

ollama_ok = is_ollama_available()
print(f'Ollama available: {ollama_ok}')

if ollama_ok:
    resp = requests.get('http://localhost:11434/api/tags')
    models = [m['name'] for m in resp.json().get('models', [])]
    print(f'Models: {models}')

Ollama available: True
Models: ['qwen2.5:7b', 'mistral:7b-instruct-q3_K_S', 'mistral:7b']


In [7]:
# Classify a single review (pick the first with text)
test_review = next((r for r in all_reviews if r.get('text')), None)
if test_review and ollama_ok:
    print(f"Review: {test_review.get('title', '(no title)')}")
    print(f"Text: {test_review['text'][:300]}...")
    print()
    topics = classify_holidaycheck_review(test_review['text'])
    print(f'Classification result ({len(topics)} topics):')
    for t in topics:
        icon = '\U0001f7e2' if t['sentiment'] == 'positive' else '\U0001f534'
        detail = f" – {t['detail']}" if t.get('detail') else ""
        print(f"  {icon} {t['topic']} = {t['sentiment']}{detail}")
else:
    print('No review with text found or Ollama not available')

Review: Modernes neues Hotel mit traumhaftem Rooftop-Pool
Text: [Allgemein] Wir haben unseren Aufenthalt im Ananea Castelo Suites Algarve im August verbracht – nur kurze Zeit nach der Eröffnung. Uns war bewusst, dass bei einer so frühen Buchung noch nicht alles perfekt eingespielt sein würde, und genau so war es auch: Es gab noch kleinere offene Baustellenbereic...

Classification result (5 topics):
  🟢 commodities = positive – modern hotel
  🟢 employees = positive – best service
  🟢 comfort = positive – good sleep quality
  🟢 cleaning = positive – clean room
  🟢 meals = positive – good food taste


In [8]:
# Classify ALL fetched reviews, save to JSON
from datetime import datetime

if ollama_ok:
    json_path = str(ROOT / 'data' / 'holidaycheck_reviews.json')
    existing = hcr.load_reviews(json_path)
    existing_by_id = {r['id']: r for r in existing}

    for r in all_reviews:
        text = r.get('text', '')
        if not text:
            continue
        topics = classify_holidaycheck_review(text)

        review_id = str(r.get('id', ''))

        record = {
            'id': review_id,
            'hotel': hcr.ANANEA_HOTEL,
            'source': 'holidaycheck',
            'rating': hcr._normalize_rating(r.get('rating')),
            'title': r.get('title', ''),
            'text': text,
            'published_date': r.get('travel_date', ''),
            'author_name': r.get('author_name', ''),
            'country': r.get('country', ''),
            'trip_type': r.get('trip_type', ''),
            'scraped_date': datetime.now().strftime('%Y-%m-%d'),
            'topics': topics,
            'classified': True,
        }

        action = 'updated' if review_id in existing_by_id else 'added'
        existing_by_id[review_id] = record

        rating = hcr._normalize_rating(r.get('rating')) or 0
        topic_str = ', '.join(f"{t['topic']}({t['sentiment'][:3]})" for t in topics)
        print(f"{rating}/6 [{action}] {r.get('title', 'N/A')[:45]:45s} → {topic_str or '(none detected)'}")

    all_records = list(existing_by_id.values())
    hcr.save_reviews(all_records, json_path)
    print(f'\nSaved {len(all_records)} reviews to {json_path}')
else:
    print('Ollama not available – skip classification')

6.0/6 [added] Modernes neues Hotel mit traumhaftem Rooftop- → commodities(pos), employees(pos), comfort(pos), cleaning(pos), meals(pos)
6.0/6 [added] insgesamt empfehlenswert                      → employees(pos)
6.0/6 [added] Neues stilvolles Hotel                        → commodities(pos), employees(pos), comfort(pos)
6.0/6 [added] Hotel kann man weiterempfehlen                → commodities(pos), employees(pos), cleaning(pos), comfort(pos), commodities(neg)
6.0/6 [added] Erholsame Urlaubstage in einem neuen schönen  → commodities(pos), employees(pos), meals(pos)
4.0/6 [added] Neues Hotel mit Potential nach Oben           → commodities(pos), employees(pos), comfort(pos), cleaning(pos), employees(neg), commodities(neg)
6.0/6 [added] Weil einfach alles gepasst hat                → employees(pos), meals(pos), commodities(pos), comfort(pos)
6.0/6 [added] Tolles Hotel, wir waren sehr zufrieden        → commodities(pos), employees(pos), cleaning(pos)
6.0/6 [added] Architektur, Stil & unverg

In [ ]:
# Debug: see the raw Ollama response for a specific review using the HC-specific prompt
DEBUG_INDEX = 0

if ollama_ok and all_reviews:
    review = all_reviews[DEBUG_INDEX]
    text = review.get('text', '')
    print(f"Review: {review.get('title', '(no title)')}")
    print(f"Full text:\n{text}\n")
    print('--- Sending to Ollama (HolidayCheck section-aware prompt) ---')

    from src.classification import _parse_classification

    # Uses classify_holidaycheck_review which understands [Section] labels
    topics = classify_holidaycheck_review(text)
    print(f'\nClassification result ({len(topics)} topics):')
    for t in topics:
        icon = '\U0001f7e2' if t['sentiment'] == 'positive' else '\U0001f534'
        detail = f" – {t['detail']}" if t.get('detail') else ""
        print(f"  {icon} {t['topic']} = {t['sentiment']}{detail}")

## 4. Manual Review Input

HolidayCheck pages may block scraping or return limited results.
Use this section to manually add reviews you found on the website.

In [ ]:
from datetime import datetime
import hashlib

# ====== FILL IN YOUR REVIEW HERE ======
manual_review = {
    'reviewer_name': 'Hans M.',                    # reviewer name (used for ID)
    'rating': 5.0,                                 # 0-6 (HolidayCheck scale)
    'title': 'Toller Urlaub',                      # review title
    'text': 'Paste the full review text here...',   # full review text
    'published_date': '2026-03-01',                 # YYYY-MM-DD
}
# ========================================

# Generate unique ID from name + date + title
id_seed = f"{manual_review['reviewer_name']}_{manual_review['published_date']}_{manual_review['title']}"
review_id = 'manual_' + hashlib.sha256(id_seed.encode()).hexdigest()[:12]

record = {
    'id': review_id,
    'hotel': hcr.ANANEA_HOTEL,
    'source': 'holidaycheck',
    'rating': manual_review['rating'],
    'title': manual_review['title'],
    'text': manual_review['text'],
    'published_date': manual_review['published_date'],
    'author_name': manual_review['reviewer_name'],
    'scraped_date': datetime.now().strftime('%Y-%m-%d'),
    'topics': [],
    'classified': False,
}

print(f'Generated ID: {review_id}')
print(json.dumps(record, indent=2, ensure_ascii=False))

In [ ]:
# Classify the manual review (optional)
if ollama_ok and record['text'] and record['text'] != 'Paste the full review text here...':
    topics = classify_holidaycheck_review(record['text'])
    record['topics'] = topics
    record['classified'] = True
    print(f'Classification ({len(topics)} topics):')
    for t in topics:
        icon = '\U0001f7e2' if t['sentiment'] == 'positive' else '\U0001f534'
        print(f"  {icon} {t['topic']} = {t['sentiment']}")
else:
    print('Ollama not available or no text – skipping classification')

In [ ]:
# Save the manual review to the JSON file
json_path = str(ROOT / 'data' / 'holidaycheck_reviews.json')
existing = hcr.load_reviews(json_path)

def _dup_key(r):
    return (r.get('author_name', r.get('reviewer_name', '')).lower().strip(), r.get('text', '')[:10].lower().strip())

existing_keys = {_dup_key(r) for r in existing}
if _dup_key(record) in existing_keys:
    print(f'⚠️  Review by {record["author_name"]} already exists (matched by name + text) – skipping')
else:
    existing.append(record)
    hcr.save_reviews(existing, json_path)
    print(f'✅ Saved! Total reviews: {len(existing)}')

## 5. Batch: Add Multiple Manual Reviews

Paste multiple reviews at once.

In [ ]:
import hashlib

batch_reviews = [
    {
        'reviewer_name': 'Maria S.',
        'rating': 4.5,
        'title': 'Schönes Hotel, kalter Pool',
        'text': 'Replace with actual review text...',
        'published_date': '2026-02-15',
    },
    # Add more reviews here...
]

json_path = str(ROOT / 'data' / 'holidaycheck_reviews.json')
existing = hcr.load_reviews(json_path)

def _dup_key(r):
    return (r.get('author_name', r.get('reviewer_name', '')).lower().strip(), r.get('text', '')[:10].lower().strip())

existing_keys = {_dup_key(r) for r in existing}
added = 0

for mr in batch_reviews:
    id_seed = f"{mr['reviewer_name']}_{mr['published_date']}_{mr['title']}"
    rid = 'manual_' + hashlib.sha256(id_seed.encode()).hexdigest()[:12]

    key = _dup_key(mr)
    if key in existing_keys:
        print(f'⚠️  Skip duplicate: {mr["title"]}')
        continue

    rec = {
        'id': rid,
        'hotel': hcr.ANANEA_HOTEL,
        'source': 'holidaycheck',
        'rating': mr['rating'],
        'title': mr['title'],
        'text': mr['text'],
        'published_date': mr.get('published_date', ''),
        'author_name': mr['reviewer_name'],
        'scraped_date': datetime.now().strftime('%Y-%m-%d'),
        'topics': [],
        'classified': False,
    }

    if ollama_ok and rec['text'] and 'Replace with' not in rec['text']:
        try:
            topics = classify_holidaycheck_review(rec['text'])
            rec['topics'] = topics
            rec['classified'] = True
        except Exception as e:
            print(f'  Classification failed: {e}')

    existing.append(rec)
    existing_keys.add(key)
    added += 1
    topic_str = ', '.join(f"{t['topic']}({t['sentiment'][:3]})" for t in rec.get('topics', []))
    print(f"✅ {mr['title']} → {topic_str or '(unclassified)'}")

if added:
    hcr.save_reviews(existing, json_path)
    print(f'\nSaved {added} new reviews. Total: {len(existing)}')
else:
    print('No new reviews to add.')

## 6. Reclassify Reviews with Empty Topics

Finds reviews that were classified but got 0 topics (LLM missed), and retries.

In [ ]:
json_path = str(ROOT / 'data' / 'holidaycheck_reviews.json')
reviews = hcr.load_reviews(json_path)

needs_retry = [r for r in reviews if r.get('classified') and not r.get('topics') and r.get('text')]
unclassified = [r for r in reviews if not r.get('classified') and r.get('text')]

print(f'Total reviews: {len(reviews)}')
print(f'Classified with 0 topics (needs retry): {len(needs_retry)}')
print(f'Unclassified: {len(unclassified)}')

to_reclassify = needs_retry + unclassified

if ollama_ok and to_reclassify:
    for r in to_reclassify:
        try:
            topics = classify_holidaycheck_review(r['text'])
            r['topics'] = topics
            r['classified'] = True
            topic_str = ', '.join(f"{t['topic']}({t['sentiment'][:3]})" for t in topics)
            print(f"  ✅ {r.get('title', 'N/A')[:40]} → {topic_str or '(still none)'}")
        except Exception as e:
            print(f"  ❌ {r.get('title', 'N/A')[:40]} → Error: {e}")

    hcr.save_reviews(reviews, json_path)
    print(f'\nDone. Saved {len(reviews)} reviews.')
elif not ollama_ok:
    print('Ollama not available')
else:
    print('Nothing to reclassify')

## 7. Reclassify ALL Reviews

Re-runs classification on **every** review with text. Use this after changing
the classification prompt in `src/classification.py`.

In [7]:
json_path = str(ROOT / 'data' / 'holidaycheck_reviews.json')
reviews = hcr.load_reviews(json_path)

with_text = [r for r in reviews if r.get('text')]
print(f'Total reviews: {len(reviews)}')
print(f'Reviews with text (will reclassify): {len(with_text)}')

if ollama_ok and with_text:
    for i, r in enumerate(with_text, 1):
        try:
            old_topics = r.get('topics', [])
            new_topics = classify_holidaycheck_review(r['text'])
            r['topics'] = new_topics
            r['classified'] = True

            old_str = ', '.join(f"{t['topic']}({t['sentiment'][:3]})" for t in old_topics)
            new_str = ', '.join(f"{t['topic']}({t['sentiment'][:3]})" for t in new_topics)
            changed = '\U0001f504' if old_str != new_str else '\u2705'
            print(f"  {changed} [{i}/{len(with_text)}] {r.get('title', 'N/A')[:40]}")
            if old_str != new_str:
                print(f"       old: {old_str or '(none)'}")
                print(f"       new: {new_str or '(none)'}")
        except Exception as e:
            print(f"  \u274c [{i}/{len(with_text)}] {r.get('title', 'N/A')[:40]} \u2192 Error: {e}")

    hcr.save_reviews(reviews, json_path)
    print(f'\nDone. Reclassified {len(with_text)} reviews. Saved {len(reviews)} total.')
elif not ollama_ok:
    print('Ollama not available')
else:
    print('No reviews with text found.')

Total reviews: 19
Reviews with text (will reclassify): 19
  ❌ [1/19] Modernes neues Hotel mit traumhaftem Roo → Error: HTTPConnectionPool(host='localhost', port=11434): Read timed out. (read timeout=180)
  ✅ [2/19] insgesamt empfehlenswert
  🔄 [3/19] Neues stilvolles Hotel
       old: commodities(pos), employees(pos), comfort(pos)
       new: commodities(pos), comfort(pos), employees(pos)
  🔄 [4/19] Hotel kann man weiterempfehlen
       old: commodities(pos), employees(pos), cleaning(pos), comfort(pos), commodities(neg)
       new: commodities(pos), cleaning(pos), employees(pos), comfort(neg), meals(pos), quality_price(neg)
  🔄 [5/19] Erholsame Urlaubstage in einem neuen sch
       old: commodities(pos), employees(pos), meals(pos)
       new: commodities(pos), meals(pos), employees(pos), commodities(neg), comfort(pos), return(pos)
  🔄 [6/19] Neues Hotel mit Potential nach Oben
       old: commodities(pos), employees(pos), comfort(pos), cleaning(pos), employees(neg), commodities(neg)
  

## 8. Current Data Summary

In [ ]:
import pandas as pd

json_path = str(ROOT / 'data' / 'holidaycheck_reviews.json')
reviews = hcr.load_reviews(json_path)

print(f'Total reviews: {len(reviews)}')
print(f'Classified: {sum(1 for r in reviews if r.get("classified"))}')
print(f'With topics: {sum(1 for r in reviews if r.get("topics"))}')
print(f'Manual: {sum(1 for r in reviews if "manual" in str(r.get("id", "")))}')
print()

# Rating distribution (0-6 scale)
ratings = [r.get('rating') for r in reviews if r.get('rating') is not None]
if ratings:
    print(f'Average rating: {sum(ratings)/len(ratings):.1f}/6')
    print(f'Rating range: {min(ratings):.1f} - {max(ratings):.1f}')
    print()

# Topic breakdown
topic_counts = {}
for r in reviews:
    for t in r.get('topics', []):
        key = f"{t['topic']} ({t['sentiment']})"
        topic_counts[key] = topic_counts.get(key, 0) + 1

if topic_counts:
    df = pd.DataFrame.from_dict(topic_counts, orient='index', columns=['Count'])
    df = df.sort_index()
    print('Topic sentiment counts:')
    print(df.to_string())
else:
    print('No topics classified yet.')